# WGAN-GP on MNIST (PyTorch)

An upgrade from the DCGAN: **Wasserstein GAN with Gradient Penalty**. It fixes the mode collapse / instability you can hit with a plain DCGAN.

Key differences from DCGAN:
- The discriminator becomes a **critic**: it outputs a raw score (no sigmoid), not a 0–1 probability.
- **Wasserstein loss** instead of BCE — the loss actually correlates with image quality.
- A **gradient penalty** keeps the critic 1-Lipschitz, which is what makes training stable.
- The critic uses **InstanceNorm** (not BatchNorm — BatchNorm breaks the gradient penalty).
- The critic is trained **`n_critic` times per generator step**.

Reference: Gulrajani et al., *Improved Training of Wasserstein GANs* (2017).

## 1. Setup & imports

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Hyperparameters

Images are kept at **32×32** (plenty for MNIST and much faster than 64×64, which matters because the critic runs 5× per step).

In [ ]:
config = dict(
    image_size=32,      # 32x32 keeps WGAN-GP fast (critic runs n_critic times/step)
    channels=1,
    z_dim=100,
    g_features=64,
    d_features=64,
    batch_size=64,     # bigger batch -> more GPU utilization, fewer CPU round-trips
    epochs=10,
    lr=1e-4,            # WGAN-GP standard learning rate
    beta1=0.0,          # WGAN-GP standard Adam betas
    beta2=0.9,
    n_critic=5,         # critic updates per generator update
    lambda_gp=10,       # gradient-penalty weight
    seed=42,
)

torch.manual_seed(config["seed"])

## 3. Data

In [ ]:
transform = transforms.Compose([
    transforms.Resize(config["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),  # [0,1] -> [-1,1]
])

dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
dataloader = DataLoader(
    dataset,
    batch_size=config["batch_size"],
    shuffle=True,
    num_workers=0,  # 0 on Windows/Jupyter to avoid the loader hanging
    pin_memory=True,
    drop_last=True,
)
print(f"Dataset size: {len(dataset)} | batches/epoch: {len(dataloader)}")

## 4. Models

**Generator** (32×32): `1 -> 4 -> 8 -> 16 -> 32`, BatchNorm + ReLU, `tanh` output.

**Critic** (32×32): strided convs `32 -> 16 -> 8 -> 4 -> 1`. Uses **InstanceNorm** and outputs a single raw score (no sigmoid).

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim, channels, g):
        super().__init__()
        self.net = nn.Sequential(
            # z_dim x 1 x 1 -> (g*4) x 4 x 4
            self._block(z_dim, g * 4, 4, 1, 0),
            # -> (g*2) x 8 x 8
            self._block(g * 4, g * 2, 4, 2, 1),
            # -> g x 16 x 16
            self._block(g * 2, g, 4, 2, 1),
            # -> channels x 32 x 32
            nn.ConvTranspose2d(g, channels, 4, 2, 1),
            nn.Tanh(),
        )

    @staticmethod
    def _block(in_c, out_c, k, s, p):
        return nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class Critic(nn.Module):
    def __init__(self, channels, d):
        super().__init__()
        self.net = nn.Sequential(
            # channels x 32 x 32 -> d x 16 x 16  (no norm on first layer)
            nn.Conv2d(channels, d, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (d*2) x 8 x 8
            self._block(d, d * 2, 4, 2, 1),
            # -> (d*4) x 4 x 4
            self._block(d * 2, d * 4, 4, 2, 1),
            # -> 1 x 1 x 1  (raw score, no sigmoid)
            nn.Conv2d(d * 4, 1, 4, 1, 0),
        )

    @staticmethod
    def _block(in_c, out_c, k, s, p):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, k, s, p, bias=False),
            nn.InstanceNorm2d(out_c, affine=True),  # InstanceNorm, NOT BatchNorm
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.net(x).view(-1)  # raw scores, shape (N,)


def weights_init(m):
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
gen = Generator(config["z_dim"], config["channels"], config["g_features"]).to(device)
critic = Critic(config["channels"], config["d_features"]).to(device)
gen.apply(weights_init)
critic.apply(weights_init)

n_params = lambda m: sum(p.numel() for p in m.parameters())
print(f"Generator params: {n_params(gen):,}")
print(f"Critic params:    {n_params(critic):,}")

## 5. Gradient penalty

Samples random points **between** real and fake images, and penalizes the critic if the norm of its gradient there strays from 1. This enforces the 1-Lipschitz constraint that makes WGAN training stable.

In [ ]:
def gradient_penalty(critic, real, fake, device):
    bs, c, h, w = real.shape
    eps = torch.rand(bs, 1, 1, 1, device=device).expand_as(real)
    interpolated = (eps * real + (1 - eps) * fake).requires_grad_(True)
    scores = critic(interpolated)
    grad = torch.autograd.grad(
        outputs=scores,
        inputs=interpolated,
        grad_outputs=torch.ones_like(scores),
        create_graph=True,
        retain_graph=True,
    )[0]
    grad = grad.view(bs, -1)
    gp = ((grad.norm(2, dim=1) - 1) ** 2).mean()
    return gp

## 6. Optimizers & wandb

In [ ]:
opt_gen = optim.Adam(gen.parameters(), lr=config["lr"], betas=(config["beta1"], config["beta2"]))
opt_critic = optim.Adam(critic.parameters(), lr=config["lr"], betas=(config["beta1"], config["beta2"]))

fixed_noise = torch.randn(64, config["z_dim"], 1, 1, device=device)

wandb.init(project="wgan-gp-mnist", config=config)

## 7. Training loop

For each batch: train the critic `n_critic` times (maximize `mean(C(real)) - mean(C(fake))` minus the gradient penalty), then train the generator once (maximize `mean(C(fake))`).

In [ ]:
step = 0
for epoch in range(config["epochs"]):
    pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{config['epochs']}", leave=True)
    for real, _ in pbar:
        real = real.to(device)
        bs = real.size(0)

        # ---- Train Critic n_critic times ----
        for _ in range(config["n_critic"]):
            noise = torch.randn(bs, config["z_dim"], 1, 1, device=device)
            fake = gen(noise)
            critic_real = critic(real)
            critic_fake = critic(fake.detach())
            gp = gradient_penalty(critic, real, fake.detach(), device)
            loss_critic = -(critic_real.mean() - critic_fake.mean()) + config["lambda_gp"] * gp

            opt_critic.zero_grad()
            loss_critic.backward()
            opt_critic.step()

        # ---- Train Generator: maximize C(G(z)) ----
        noise = torch.randn(bs, config["z_dim"], 1, 1, device=device)
        fake = gen(noise)
        loss_gen = -critic(fake).mean()

        opt_gen.zero_grad()
        loss_gen.backward()
        opt_gen.step()

        # ---- Logging ----
        if step % 50 == 0:
            # Wasserstein distance estimate = C(real) - C(fake); higher loss_critic magnitude tracks quality
            w_dist = (critic_real.mean() - critic_fake.mean()).item()
            pbar.set_postfix(loss_c=f"{loss_critic.item():.2f}",
                             loss_g=f"{loss_gen.item():.2f}",
                             W_dist=f"{w_dist:.2f}")
            wandb.log({
                "loss/critic": loss_critic.item(),
                "loss/generator": loss_gen.item(),
                "wasserstein_distance": w_dist,
                "gradient_penalty": gp.item(),
                "epoch": epoch,
            }, step=step)
        step += 1

    # ---- End of epoch: log + save a grid of samples ----
    gen.eval()
    with torch.no_grad():
        samples = gen(fixed_noise)
    gen.train()
    grid = make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1))
    wandb.log({"generated": wandb.Image(grid, caption=f"epoch {epoch + 1}")}, step=step)

    os.makedirs("checkpoints", exist_ok=True)
    torch.save(gen.state_dict(), f"checkpoints/wgan_gen_ep{epoch + 1}.pth")

print("Training complete.")

## 8. Save & finish

In [ ]:
torch.save(gen.state_dict(), "checkpoints/wgan_generator.pth")
torch.save(critic.state_dict(), "checkpoints/wgan_critic.pth")
wandb.finish()
print("Saved models to ./checkpoints/ (per-epoch generators saved too, so you can pick the best one)")

## 9. Generate samples from the trained model

In [ ]:
import matplotlib.pyplot as plt

gen.eval()
with torch.no_grad():
    noise = torch.randn(64, config["z_dim"], 1, 1, device=device)
    samples = gen(noise)
grid = make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1)).cpu()

plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("WGAN-GP generated MNIST digits")
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.show()